[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/13_vision_transformer_blocks.ipynb)

# 13. Vision transformer and detection blocks — paper-faithful core structures

이전 버전은 Swin에서 window를 자르기만 하고 **shifted window와 relative position bias**가 없었고, FPN은 lateral projection 없이 feature를 그냥 더했으며, center decode도 heatmap index만 읽었다.

이번 버전은 ViT → Swin → FPN → CenterNet식 anchor-free decode의 중요한 계산 그래프를 유지한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. ViT patch embedding + position + CLS

ViT는 patchify만으로 끝나지 않는다. patch projection 뒤 CLS token과 position embedding을 더한 token sequence가 Transformer encoder로 들어간다.


In [ ]:
image = torch.randn(
    2, 3, 16, 16,
    device=device,
)
patch_size = 4
hidden_dim = 24

patches = F.unfold(
    image,
    kernel_size=patch_size,
    stride=patch_size,
).transpose(1, 2)

patch_projection = nn.Linear(
    3 * patch_size * patch_size,
    hidden_dim,
).to(device)

patch_tokens = patch_projection(patches)
num_patches = patch_tokens.size(1)

cls_token = nn.Parameter(
    torch.zeros(1, 1, hidden_dim, device=device)
)
position_embedding = nn.Parameter(
    torch.randn(1, 1 + num_patches, hidden_dim, device=device) * 0.02
)

tokens = torch.cat(
    [cls_token.expand(image.size(0), -1, -1), patch_tokens],
    dim=1,
)
tokens = tokens + position_embedding

encoder = nn.TransformerEncoderLayer(
    d_model=hidden_dim,
    nhead=3,
    dim_feedforward=4 * hidden_dim,
    batch_first=True,
).to(device)

encoded = encoder(tokens)
print("ViT tokens:", encoded.shape)


## 2. Swin window attention with relative position bias

Swin의 window attention은 단순 local mask가 아니다. 각 window 안에서 attention하고, 상대 위치 `(Δy,Δx)`마다 learnable bias를 attention score에 더한다.


In [ ]:
def window_partition(x, window_size):
    batch_size, height, width, channels = x.shape

    x = x.view(
        batch_size,
        height // window_size,
        window_size,
        width // window_size,
        window_size,
        channels,
    )
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()

    return x.view(-1, window_size * window_size, channels)


def relative_position_index(window_size, device):
    coordinates = torch.stack(
        torch.meshgrid(
            torch.arange(window_size, device=device),
            torch.arange(window_size, device=device),
            indexing="ij",
        )
    )
    coordinates = coordinates.flatten(1)

    relative = coordinates[:, :, None] - coordinates[:, None, :]
    relative = relative.permute(1, 2, 0).contiguous()

    relative[:, :, 0] += window_size - 1
    relative[:, :, 1] += window_size - 1
    relative[:, :, 0] *= 2 * window_size - 1

    return relative.sum(dim=-1)


feature = torch.randn(
    1, 4, 4, 8,
    device=device,
)
window_size = 2
windows = window_partition(feature, window_size)

num_heads = 2
head_dim = 4
qkv_projection = nn.Linear(8, 3 * 8).to(device)

qkv = qkv_projection(windows)
qkv = qkv.view(
    windows.size(0),
    windows.size(1),
    3,
    num_heads,
    head_dim,
).permute(2, 0, 3, 1, 4)
q, k, v = qkv.unbind(0)

scores = q @ k.transpose(-2, -1)
scores = scores / (head_dim ** 0.5)

bias_table = nn.Parameter(
    torch.zeros(
        (2 * window_size - 1) ** 2,
        num_heads,
        device=device,
    )
)
relative_index = relative_position_index(
    window_size,
    device,
)

relative_bias = bias_table[relative_index.reshape(-1)]
relative_bias = relative_bias.view(
    window_size * window_size,
    window_size * window_size,
    num_heads,
).permute(2, 0, 1)

scores = scores + relative_bias[None]
weights = scores.softmax(dim=-1)
window_output = weights @ v

print("windows:", windows.shape)
print("relative bias:", relative_bias.shape)
print("window attention:", window_output.shape)


## 3. Shifted windows connect neighboring windows

Swin은 다음 block에서 feature map을 cyclic shift한 뒤 다시 window partition한다. 그래서 이전 block에서 서로 다른 window에 있던 token들이 다음 block에서는 같은 window에서 attention할 수 있다.


In [ ]:
shift_size = window_size // 2

shifted_feature = torch.roll(
    feature,
    shifts=(-shift_size, -shift_size),
    dims=(1, 2),
)
shifted_windows = window_partition(
    shifted_feature,
    window_size,
)

print("normal first window token ids would be local")
print("shifted windows shape:", shifted_windows.shape)
print("shift size:", shift_size)


## 4. Patch merging

Swin stage 사이에서는 2×2 neighboring tokens를 channel 방향으로 concat하고 LayerNorm + linear reduction으로 spatial resolution을 절반으로 줄이면서 channel dimension을 늘린다.


In [ ]:
x = torch.randn(1, 4, 4, 8, device=device)

merged = torch.cat(
    [
        x[:, 0::2, 0::2],
        x[:, 1::2, 0::2],
        x[:, 0::2, 1::2],
        x[:, 1::2, 1::2],
    ],
    dim=-1,
)

merge_norm = nn.LayerNorm(32).to(device)
merge_reduction = nn.Linear(32, 16, bias=False).to(device)

merged_output = merge_reduction(merge_norm(merged))
print("patch merged:", merged_output.shape)


## 5. FPN: lateral projection + top-down fusion + smoothing

FPN은 서로 channel 수가 다른 backbone stage를 그대로 더하지 않는다. 각 stage에 1×1 lateral projection으로 같은 channel dimension을 만든 뒤, top-down upsample과 합치고 3×3 convolution으로 최종 pyramid feature를 만든다.


In [ ]:
c3 = torch.randn(1, 32, 16, 16, device=device)
c4 = torch.randn(1, 64, 8, 8, device=device)
c5 = torch.randn(1, 128, 4, 4, device=device)

lateral3 = nn.Conv2d(32, 24, 1).to(device)
lateral4 = nn.Conv2d(64, 24, 1).to(device)
lateral5 = nn.Conv2d(128, 24, 1).to(device)

smooth3 = nn.Conv2d(24, 24, 3, padding=1).to(device)
smooth4 = nn.Conv2d(24, 24, 3, padding=1).to(device)
smooth5 = nn.Conv2d(24, 24, 3, padding=1).to(device)

p5_inner = lateral5(c5)
p4_inner = lateral4(c4) + F.interpolate(
    p5_inner,
    size=c4.shape[-2:],
    mode="nearest",
)
p3_inner = lateral3(c3) + F.interpolate(
    p4_inner,
    size=c3.shape[-2:],
    mode="nearest",
)

p5 = smooth5(p5_inner)
p4 = smooth4(p4_inner)
p3 = smooth3(p3_inner)

print("P3:", p3.shape)
print("P4:", p4.shape)
print("P5:", p5.shape)


## 6. CenterNet-style anchor-free decode

center heatmap의 peak만 고르는 것으로 box가 완성되지 않는다. center index에 sub-pixel offset을 더하고 width/height head를 읽어 bounding box를 복원한다.


In [ ]:
heatmap = torch.tensor(
    [[[[0.1, 0.7], [0.2, 0.9]]]],
    device=device,
)
offset = torch.tensor(
    [
        [
            [[0.1, 0.2], [0.0, -0.1]],
            [[0.0, 0.1], [0.2, 0.3]],
        ]
    ],
    device=device,
)
size = torch.tensor(
    [
        [
            [[2.0, 2.5], [1.5, 3.0]],
            [[1.0, 1.5], [2.0, 2.5]],
        ]
    ],
    device=device,
)

score, flat_index = heatmap.flatten(2).topk(1, dim=-1)
height, width = heatmap.shape[-2:]

center_y = flat_index // width
center_x = flat_index % width

x_index = int(center_x.item())
y_index = int(center_y.item())

dx = offset[0, 0, y_index, x_index]
dy = offset[0, 1, y_index, x_index]
box_width = size[0, 0, y_index, x_index]
box_height = size[0, 1, y_index, x_index]

center_x_float = center_x.float() + dx
center_y_float = center_y.float() + dy

x1 = center_x_float - box_width / 2
y1 = center_y_float - box_height / 2
x2 = center_x_float + box_width / 2
y2 = center_y_float + box_height / 2

box = torch.stack([x1, y1, x2, y2], dim=-1)

print("score:", score.flatten())
print("decoded box:", box)


## References and provenance

**ViT** — Dosovitskiy et al. patch projection, CLS token, positional embedding, encoder를 반영했다.

**Swin Transformer** — Liu et al. non-overlapping window attention, relative position bias, shifted-window connectivity, patch merging을 반영했다. 실제 Swin은 shifted-window boundary를 위한 attention mask도 사용한다.

**FPN** — Lin et al. 1×1 lateral projection, top-down upsampling, elementwise fusion, 3×3 output convolution을 반영했다.

**CenterNet** — Objects as Points 계열의 center heatmap + offset + size decode를 반영했다. YOLO/RT-DETR은 별도 detection family이므로 이 decode와 같은 구조라고 뭉뚱그리지 않는다.
